# Bronze

- Reads the raw datasets from the Unity Catalog volume `crop_risk.source_data.raw` and writes them into bronze Delta tables, except for audit columns.

In [0]:
CATALOG = "crop_risk"
VOLUME_PATH = f"/Volumes/{CATALOG}/source_data/raw"

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, BooleanType
import pyspark.sql.functions as F

`crop_trend_master`

In [0]:
crop_trend_schema = StructType([
    StructField("province", StringType(), True),
    StructField("year", IntegerType(), True),
    StructField("quarter", StringType(), True),
    StructField("temperature_2m_mean", DoubleType(), True),
    StructField("temperature_2m_max", DoubleType(), True),
    StructField("temperature_2m_min", DoubleType(), True),
    StructField("precipitation_sum", DoubleType(), True),
    StructField("rain_sum", DoubleType(), True),
    StructField("precipitation_hours", IntegerType(), True),
    StructField("sunshine_duration", DoubleType(), True),
    StructField("shortwave_radiation_sum", DoubleType(), True),
    StructField("reference_evapotranspiration_mm", DoubleType(), True),
    StructField("wind_speed_10m_max", DoubleType(), True),
    StructField("wind_gusts_10m_max", DoubleType(), True),
    StructField("rain_normal", DoubleType(), True),
    StructField("temp_normal", DoubleType(), True),
    StructField("rainfall_deviation_pct", DoubleType(), True),
    StructField("temperature_anomaly_c", DoubleType(), True),
    StructField("oni_index", DoubleType(), True),
    StructField("crop", StringType(), True),
    StructField("production", DoubleType(), True),
    StructField("quarter_num", IntegerType(), True),
    StructField("time_index", IntegerType(), True),
    StructField("production_lag_1", DoubleType(), True),
    StructField("production_lag_2", DoubleType(), True),
    StructField("production_lag_4", DoubleType(), True),
    StructField("temp_anomaly_rolling", DoubleType(), True),
    StructField("rain_anomaly_rolling", DoubleType(), True),
    StructField("enso_phase_la_nina", BooleanType(), True),
    StructField("enso_phase_neutral", BooleanType(), True),
    StructField("crop_group", StringType(), True),
    StructField("growth_rate_yoy", DoubleType(), True),
    StructField("climate_stress", DoubleType(), True),
    StructField("risk_label", StringType(), True),
])

In [0]:
df_crop_trend = (
    spark.read
    .option("header", True)
    .option("delimiter", ",")
    .schema(crop_trend_schema)
    .csv(f"{VOLUME_PATH}/crop_trend_master.csv")
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
)

(
    df_crop_trend.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.bronze.crop_trend_master")
)

print(f"crop_trend_master: {df_crop_trend.count():,} rows")

`province_weather_quarterly`

In [0]:
weather_schema = StructType([
    StructField("province", StringType(), True),
    StructField("year", IntegerType(), True),
    StructField("year_quarter", StringType(), True),
    StructField("temperature_2m_mean", DoubleType(), True),
    StructField("temperature_2m_max", DoubleType(), True),
    StructField("temperature_2m_min", DoubleType(), True),
    StructField("precipitation_sum", DoubleType(), True),
    StructField("rain_sum", DoubleType(), True),
    StructField("precipitation_hours", IntegerType(), True),
    StructField("sunshine_duration", DoubleType(), True),
    StructField("shortwave_radiation_sum", DoubleType(), True),
    StructField("reference_evapotranspiration_mm", DoubleType(), True),
    StructField("wind_speed_10m_max", DoubleType(), True),
    StructField("wind_gusts_10m_max", DoubleType(), True),
    StructField("quarter", StringType(), True),
    StructField("rain_normal", DoubleType(), True),
    StructField("temp_normal", DoubleType(), True),
    StructField("rainfall_deviation_pct", DoubleType(), True),
    StructField("temperature_anomaly_c", DoubleType(), True),
    StructField("oni_index", DoubleType(), True),
    StructField("enso_phase", StringType(), True),
])

In [0]:
df_weather = (
    spark.read
    .option("header", True)
    .option("delimiter", ",")
    .schema(weather_schema)
    .csv(f"{VOLUME_PATH}/province_weather_quarterly.csv")
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
)

(
    df_weather.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.bronze.province_weather_quarterly")
)

print(f"province_weather_quarterly: {df_weather.count():,} rows")

`data_dictionary`

In [0]:
dictionary_schema = StructType([
    StructField("file", StringType(), True),
    StructField("column_name", StringType(), True),
    StructField("data_type", StringType(), True),
    StructField("unit", StringType(), True),
    StructField("derived", StringType(), True),
    StructField("description", StringType(), True),
])

In [0]:
df_dictionary = (
    spark.read
    .option("header", True)
    .option("delimiter", ",")
    .schema(dictionary_schema)
    .csv(f"{VOLUME_PATH}/data_dictionary.csv")
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
)

(
    df_dictionary.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.bronze.data_dictionary")
)

print(f"data_dictionary: {df_dictionary.count():,} rows")

Sanity Check

In [0]:
"""

for delta_table in ["crop_trend_master", "province_weather_quarterly", "data_dictionary"]:
    table_name = f"{CATALOG}.bronze.{delta_table}"
    column_count = len(spark.catalog.listColumns(table_name))
    row_count = spark.table(table_name).count()
    
    print(f"{table_name}: {column_count} columns, {row_count:,} rows")
    
"""